# 04 - Fine-tuning IndoBERT (Fase 6)

## Setup

In [1]:
#!pip install -q transformers accelerate

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import gc
import hashlib
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import KFold, StratifiedKFold
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
assert torch.cuda.is_available(), 'GPU tidak aktif: pilih Runtime > Change runtime type > T4 GPU'
PERANGKAT = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))

FOLDER_PROJECT = Path('/content/drive/MyDrive/ulasan-ewallet')
FOLDER_PROCESSED = FOLDER_PROJECT / 'data' / 'processed'
FOLDER_GOLD = FOLDER_PROJECT / 'data' / 'gold'
FOLDER_MODEL = FOLDER_PROJECT / 'models'
FOLDER_HASIL = FOLDER_PROCESSED / 'indobert'
FOLDER_HASIL.mkdir(parents=True, exist_ok=True)
SEED = 42

ASPEK = ['transaksi', 'keamanan', 'akun', 'error_performa', 'cs', 'biaya_promo', 'paylater', 'kemudahan']
KELAS = ['neg', 'net', 'pos']
INDEKS_KELAS = {k: i for i, k in enumerate(KELAS)}
RATING_KE_SENTIMEN = {1: 'neg', 2: 'neg', 3: 'net', 4: 'pos', 5: 'pos'}

ulasan = pd.read_parquet(
    FOLDER_PROCESSED / 'ulasan_bersih.parquet', columns=['id_ulasan', 'aplikasi', 'rating', 'teks_ringan']
)
gold_train = pd.read_parquet(FOLDER_GOLD / 'gold_train.parquet').merge(
    ulasan[['id_ulasan', 'teks_ringan']], on='id_ulasan', how='left'
)
assert gold_train['teks_ringan'].notna().all(), 'Ada ulasan gold yang tidak ditemukan di ulasan_bersih'
# Versi label: berubah otomatis kalau label gold train diubah (mis. setelah audit), jadi hasil lama tidak dipakai
VERSI_LABEL = hashlib.md5(
    gold_train.sort_values('id_label')[['id_label', 'sentimen'] + ASPEK].to_csv(index=False).encode()
).hexdigest()[:8]
dev = gold_train[gold_train['sumber'] != 'tambahan'].reset_index(drop=True)
tambahan = gold_train[gold_train['sumber'] == 'tambahan'].reset_index(drop=True)

# Teks test: hanya untuk diprediksi, labelnya tidak dibaca di notebook ini
kunci_gold = pd.read_parquet(FOLDER_GOLD / 'gold_kunci.parquet')
test = kunci_gold.loc[kunci_gold['bagian'] == 'test', ['id_ulasan']].merge(
    ulasan[['id_ulasan', 'teks_ringan']], on='id_ulasan', how='left'
).reset_index(drop=True)
assert test['teks_ringan'].notna().all()

# Data berlabel rating: seluruh ulasan bersih dikurangi SEMUA ulasan yang pernah dilabel (test, train, pilot, tambahan)
id_berlabel = set()
for nama_file in ['gold_kunci.parquet', 'pilot_kunci.parquet', 'tambahan_kunci.parquet']:
    id_berlabel |= set(pd.read_parquet(FOLDER_GOLD / nama_file)['id_ulasan'])
lemah = ulasan[~ulasan['id_ulasan'].isin(id_berlabel)].reset_index(drop=True)
lemah['sentimen'] = lemah['rating'].map(RATING_KE_SENTIMEN)

print(f'versi label: {VERSI_LABEL} | dev: {len(dev)} | tambahan: {len(tambahan)} | test (teks saja): {len(test)} | berlabel rating: {len(lemah):,}')

GPU: Tesla T4
versi label: 6b6ec0d2 | dev: 855 | tambahan: 153 | test (teks saja): 300 | berlabel rating: 397,434


In [4]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=KELAS, average='macro', zero_division=0)


def macro_f1_aspek(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


def interval_bootstrap(y_true, y_pred, fungsi, n=1000):
    """Interval kepercayaan 95% dengan resampling baris (berlaku untuk label tunggal maupun multi-label)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    indeks = np.random.default_rng(SEED).integers(0, len(y_true), size=(n, len(y_true)))
    return np.percentile([fungsi(y_true[i], y_pred[i]) for i in indeks], [2.5, 97.5])


def label_dari_prob(prob):
    return np.array(KELAS)[prob.argmax(axis=1)]


hasil_sentimen = []


def catat_sentimen(nama, y_pred, catatan=''):
    """Nilai prediksi sentimen di dev, simpan ke tabel ringkasan, tampilkan confusion matrix."""
    y_true, y_pred = dev['sentimen'].to_numpy(), np.asarray(y_pred)
    skor = macro_f1(y_true, y_pred)
    bawah, atas = interval_bootstrap(y_true, y_pred, macro_f1)
    f1_kelas = f1_score(y_true, y_pred, labels=KELAS, average=None, zero_division=0)
    hasil_sentimen[:] = [h for h in hasil_sentimen if h['model'] != nama]
    hasil_sentimen.append({
        'model': nama, 'macro_f1': round(skor, 3), 'ci_95': f'{bawah:.2f}-{atas:.2f}',
        **{f'f1_{k}': round(v, 3) for k, v in zip(KELAS, f1_kelas)}, 'catatan': catatan,
    })
    print(f'{nama}: macro-F1 {skor:.3f} (CI 95% {bawah:.2f}-{atas:.2f})')
    display(pd.crosstab(y_true, y_pred, rownames=['gold'], colnames=['prediksi']))


lipatan = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # sama dengan 03_baseline
lipatan_aspek = KFold(n_splits=5, shuffle=True, random_state=SEED)      # sama dengan 03_baseline

### Fungsi fine-tuning

In [5]:
MODEL_DASAR = {
    'IndoBERT': 'indobenchmark/indobert-base-p1',
    'IndoBERTweet': 'indolem/indobertweet-base-uncased',
}
MAX_LEN, UKURAN_BATCH, LR = 128, 32, 2e-5
EPOCH_GOLD, EPOCH_RATING, EPOCH_ASPEK = 5, 1, 8


def atur_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def bersihkan():
    gc.collect()
    torch.cuda.empty_cache()


def buat_loader(tokenizer, teks, label=None, acak=False, ukuran=UKURAN_BATCH, seed=SEED):
    """Tokenisasi per batch dengan padding seperlunya (lebih cepat daripada padding ke 128 untuk semua)."""
    def gabung(indeks):
        batch = tokenizer([teks[i] for i in indeks], truncation=True, max_length=MAX_LEN,
                          padding=True, return_tensors='pt')
        if label is not None:
            batch['labels'] = torch.as_tensor(label[indeks])
        return batch
    return DataLoader(list(range(len(teks))), batch_size=ukuran, shuffle=acak, collate_fn=gabung,
                      generator=torch.Generator().manual_seed(seed))


def latih(sumber, teks, label, multi_label=False, epoch=EPOCH_GOLD, lr=LR, bobot=None, seed=SEED):
    """Fine-tune dari `sumber` (nama model Hugging Face atau folder model). Mengembalikan (model, tokenizer)."""
    atur_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(sumber)
    jumlah_label = label.shape[1] if multi_label else len(KELAS)
    model = AutoModelForSequenceClassification.from_pretrained(sumber, num_labels=jumlah_label).to(PERANGKAT)
    loader = buat_loader(tokenizer, list(teks), label, acak=True, seed=seed)
    langkah_total = len(loader) * epoch
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    jadwal = get_linear_schedule_with_warmup(optimizer, int(0.1 * langkah_total), langkah_total)
    fungsi_loss = (torch.nn.BCEWithLogitsLoss(pos_weight=bobot) if multi_label
                   else torch.nn.CrossEntropyLoss(weight=bobot))
    scaler = torch.amp.GradScaler('cuda')
    model.train()
    for _ in range(epoch):
        for batch in tqdm(loader, leave=False, disable=len(loader) < 200):
            y = batch.pop('labels').to(PERANGKAT)
            batch = {k: v.to(PERANGKAT) for k, v in batch.items()}
            with torch.autocast('cuda', dtype=torch.float16):
                logit = model(**batch).logits
            loss = fungsi_loss(logit.float(), y.float() if multi_label else y)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            jadwal.step()
    return model, tokenizer


@torch.no_grad()
def prediksi(model, tokenizer, teks, multi_label=False):
    """Probabilitas per kelas (softmax) atau per aspek (sigmoid)."""
    model.eval()
    hasil = []
    for batch in buat_loader(tokenizer, list(teks), ukuran=128):
        batch = {k: v.to(PERANGKAT) for k, v in batch.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            logit = model(**batch).logits.float()
        hasil.append(torch.sigmoid(logit) if multi_label else torch.softmax(logit, dim=-1))
    return torch.cat(hasil).cpu().numpy()


def bobot_kelas(y_indeks):
    jumlah = np.bincount(y_indeks, minlength=len(KELAS))
    return torch.tensor(len(y_indeks) / (len(KELAS) * jumlah), dtype=torch.float, device=PERANGKAT)


def bobot_aspek(Y):
    positif = Y.sum(axis=0)
    return torch.tensor(np.minimum((len(Y) - positif) / np.maximum(positif, 1), 20),
                        dtype=torch.float, device=PERANGKAT)


def muat_atau_hitung(nama, fungsi):
    """Hasil disimpan ke Drive; kalau sudah ada (mis. setelah runtime terputus), dimuat tanpa melatih ulang."""
    path = FOLDER_HASIL / f'{nama}_{VERSI_LABEL}.npy'
    if path.exists():
        print(f'{nama}: dimuat dari Drive')
        return np.load(path)
    mulai = time.time()
    hasil = fungsi()
    np.save(path, hasil)
    print(f'{nama}: selesai dalam {(time.time() - mulai) / 60:.1f} menit')
    return hasil


def simpan_prediksi_test(tugas, nama_model, prob, kolom):
    """Probabilitas test disimpan sekarang; skornya dihitung setelah label test final."""
    path = FOLDER_HASIL / f'prediksi_test_{tugas}.parquet'
    baru = pd.DataFrame(prob, columns=kolom).assign(id_ulasan=test['id_ulasan'].to_numpy(), model=nama_model,
                                                 versi_label=VERSI_LABEL)
    if path.exists():
        lama = pd.read_parquet(path)
        baru = pd.concat([lama[lama['model'] != nama_model], baru], ignore_index=True)
    baru.to_parquet(path, index=False)


def sudah_diprediksi(tugas, nama_model):
    path = FOLDER_HASIL / f'prediksi_test_{tugas}.parquet'
    if not path.exists():
        return False
    ada = pd.read_parquet(path)
    if 'versi_label' not in ada.columns:
        return False
    return bool(((ada['model'] == nama_model) & (ada['versi_label'] == VERSI_LABEL)).any())


def simpan_model(model, tokenizer, folder, info):
    folder.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(folder)
    tokenizer.save_pretrained(folder)
    info = {**info, 'versi_label': VERSI_LABEL}
    (folder / 'info.json').write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding='utf-8')


def model_terbaru(folder):
    """True kalau model di folder dilatih dengan versi label yang sekarang."""
    path = folder / 'info.json'
    return path.exists() and json.loads(path.read_text(encoding='utf-8')).get('versi_label') == VERSI_LABEL


KOLOM_PROB_SENTIMEN = [f'prob_{k}' for k in KELAS]
KOLOM_PROB_ASPEK = [f'prob_{a}' for a in ASPEK]
y_dev = dev['sentimen'].map(INDEKS_KELAS).to_numpy()
y_tambahan = tambahan['sentimen'].map(INDEKS_KELAS).to_numpy()
y_gold_train = gold_train['sentimen'].map(INDEKS_KELAS).to_numpy()

tokenizer_cek = AutoTokenizer.from_pretrained(MODEL_DASAR['IndoBERT'])
panjang_token = np.array([len(tokenizer_cek(t)['input_ids']) for t in dev['teks_ringan']])
print(f'Panjang token dev: median {np.median(panjang_token):.0f}, persentil 95 {np.percentile(panjang_token, 95):.0f}, '
      f'terpotong di {MAX_LEN}: {(panjang_token > MAX_LEN).mean():.1%}')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Panjang token dev: median 12, persentil 95 52, terpotong di 128: 0.0%


# A. Sentimen
## 6.8 Pilih backbone: IndoBERT vs IndoBERTweet (label gold, 5-fold)

In [6]:
def cv_sentimen(sumber, epoch=EPOCH_GOLD):
    """5-fold di dev; sampel terarah selalu ikut di lipatan latih. Mengembalikan probabilitas out-of-fold."""
    prob = np.zeros((len(dev), len(KELAS)))
    for k, (latih_idx, uji_idx) in enumerate(lipatan.split(dev, dev['sentimen'])):
        teks_latih = dev['teks_ringan'].iloc[latih_idx].tolist() + tambahan['teks_ringan'].tolist()
        y_latih = np.concatenate([y_dev[latih_idx], y_tambahan])
        model, tok = latih(sumber, teks_latih, y_latih, epoch=epoch, bobot=bobot_kelas(y_latih), seed=SEED + k)
        prob[uji_idx] = prediksi(model, tok, dev['teks_ringan'].iloc[uji_idx].tolist())
        del model
        bersihkan()
        print(f'  lipatan {k + 1}/5 selesai')
    return prob


prob_oof = {}
for nama, id_model in MODEL_DASAR.items():
    prob_oof[f'{nama}, label gold'] = muat_atau_hitung(f'oof_sentimen_gold_{nama}', lambda: cv_sentimen(id_model))
    catat_sentimen(f'{nama}, label gold', label_dari_prob(prob_oof[f'{nama}, label gold']),
                   '5-fold di dev + sampel terarah')

DASAR_TERPILIH = max(MODEL_DASAR, key=lambda n: macro_f1(dev['sentimen'], label_dari_prob(prob_oof[f'{n}, label gold'])))
print(f'\nBackbone terpilih: {DASAR_TERPILIH} ({MODEL_DASAR[DASAR_TERPILIH]})')

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

  lipatan 1/5 selesai


... (26 outputs left out)

/tmp/ipykernel_549/2016421923.py:60: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  jadwal.step()


  lipatan 5/5 selesai
oof_sentimen_gold_IndoBERTweet: selesai dalam 1.7 menit
IndoBERTweet, label gold: macro-F1 0.831 (CI 95% 0.79-0.87)


prediksi,neg,net,pos
gold,,,
neg,551,22,13
net,15,37,7
pos,7,3,200



Backbone terpilih: IndoBERT (indobenchmark/indobert-base-p1)


## 6.9 IndoBERT berlabel rating

In [7]:
UKURAN_RATING_PER_APLIKASI = 20_000
PATH_MODEL_RATING = FOLDER_MODEL / 'indobert_sentimen_rating'
NAMA_RATING = f'{DASAR_TERPILIH}, label rating'

if (PATH_MODEL_RATING / 'info.json').exists():
    print('Model berlabel rating dimuat dari Drive')
    tok_rating = AutoTokenizer.from_pretrained(PATH_MODEL_RATING)
    model_rating = AutoModelForSequenceClassification.from_pretrained(PATH_MODEL_RATING).to(PERANGKAT)
else:
    subset_rating = (
        lemah.groupby('aplikasi', group_keys=False)
        .apply(lambda d: d.sample(n=min(UKURAN_RATING_PER_APLIKASI, len(d)), random_state=SEED))
        .reset_index(drop=True)
    )
    y_rating = subset_rating['sentimen'].map(INDEKS_KELAS).to_numpy()
    print('Subset berlabel rating:', subset_rating.groupby('aplikasi').size().to_dict(),
          '| sentimen (%):', subset_rating['sentimen'].value_counts(normalize=True).mul(100).round(1).to_dict())
    mulai = time.time()
    model_rating, tok_rating = latih(MODEL_DASAR[DASAR_TERPILIH], subset_rating['teks_ringan'].tolist(), y_rating,
                                     epoch=EPOCH_RATING, bobot=bobot_kelas(y_rating))
    print(f'Selesai dalam {(time.time() - mulai) / 60:.1f} menit')
    simpan_model(model_rating, tok_rating, PATH_MODEL_RATING, {
        'nama': NAMA_RATING, 'dasar': MODEL_DASAR[DASAR_TERPILIH], 'label': 'rating',
        'jumlah_latih': len(subset_rating), 'epoch': EPOCH_RATING, 'lr': LR, 'max_len': MAX_LEN,
    })

prob_rating_dev = prediksi(model_rating, tok_rating, dev['teks_ringan'].tolist())
np.save(FOLDER_HASIL / 'dev_sentimen_rating.npy', prob_rating_dev)
catat_sentimen(NAMA_RATING, label_dari_prob(prob_rating_dev), f'{UKURAN_RATING_PER_APLIKASI:,} ulasan per aplikasi')
if not sudah_diprediksi('sentimen', NAMA_RATING):
    simpan_prediksi_test('sentimen', NAMA_RATING, prediksi(model_rating, tok_rating, test['teks_ringan'].tolist()),
                         KOLOM_PROB_SENTIMEN)
del model_rating
bersihkan()

Model berlabel rating dimuat dari Drive


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

IndoBERT, label rating: macro-F1 0.558 (CI 95% 0.53-0.59)


prediksi,neg,net,pos
gold,,,
neg,302,242,42
net,2,26,31
pos,0,5,205


## 6.10 Dua tahap: rating lalu gold (opsional)

In [8]:
JALANKAN_DUA_TAHAP = True
NAMA_DUA_TAHAP = 'Dua tahap (rating -> gold)'

if JALANKAN_DUA_TAHAP:
    prob_oof[NAMA_DUA_TAHAP] = muat_atau_hitung('oof_sentimen_dua_tahap', lambda: cv_sentimen(str(PATH_MODEL_RATING)))
    catat_sentimen(NAMA_DUA_TAHAP, label_dari_prob(prob_oof[NAMA_DUA_TAHAP]), '5-fold di dev + sampel terarah')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  lipatan 1/5 selesai


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  lipatan 2/5 selesai


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  lipatan 3/5 selesai


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  lipatan 4/5 selesai


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  lipatan 5/5 selesai
oof_sentimen_dua_tahap: selesai dalam 1.7 menit
Dua tahap (rating -> gold): macro-F1 0.868 (CI 95% 0.83-0.90)


prediksi,neg,net,pos
gold,,,
neg,562,19,5
net,18,40,1
pos,7,0,203


## 6.11 Model sentimen final

In [9]:
PATH_MODEL_SENTIMEN = FOLDER_MODEL / 'indobert_sentimen'
konfigurasi_final = {f'{DASAR_TERPILIH}, label gold': MODEL_DASAR[DASAR_TERPILIH]}
if JALANKAN_DUA_TAHAP:
    konfigurasi_final[NAMA_DUA_TAHAP] = str(PATH_MODEL_RATING)

skor_dev = {nama: macro_f1(dev['sentimen'], label_dari_prob(prob_oof[nama])) for nama in konfigurasi_final}
TERBAIK = max(skor_dev, key=skor_dev.get)
print('Skor dev:', {n: round(s, 3) for n, s in skor_dev.items()}, '| terbaik:', TERBAIK)

for nama, sumber in konfigurasi_final.items():
    perlu_simpan = nama == TERBAIK and not model_terbaru(PATH_MODEL_SENTIMEN)
    if sudah_diprediksi('sentimen', nama) and not perlu_simpan:
        print(f'{nama}: prediksi test sudah ada, dilewati')
        continue
    mulai = time.time()
    model, tok = latih(sumber, gold_train['teks_ringan'].tolist(), y_gold_train, bobot=bobot_kelas(y_gold_train))
    simpan_prediksi_test('sentimen', nama, prediksi(model, tok, test['teks_ringan'].tolist()), KOLOM_PROB_SENTIMEN)
    if nama == TERBAIK:
        simpan_model(model, tok, PATH_MODEL_SENTIMEN, {
            'nama': nama, 'sumber': sumber, 'label': 'gold', 'jumlah_latih': len(gold_train),
            'macro_f1_dev_5fold': round(skor_dev[nama], 3), 'epoch': EPOCH_GOLD, 'lr': LR, 'max_len': MAX_LEN,
        })
        print(f'{nama}: disimpan ke models/indobert_sentimen/')
    print(f'{nama}: selesai dalam {(time.time() - mulai) / 60:.1f} menit')
    del model
    bersihkan()

Skor dev: {'IndoBERT, label gold': 0.859, 'Dua tahap (rating -> gold)': 0.868} | terbaik: Dua tahap (rating -> gold)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

IndoBERT, label gold: selesai dalam 0.4 menit


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Dua tahap (rating -> gold): disimpan ke models/indobert_sentimen/
Dua tahap (rating -> gold): selesai dalam 0.4 menit


# B. Aspek (multi-label)
## 6.12 IndoBERT multi-label (5-fold)

In [10]:
Y_dev = dev[ASPEK].to_numpy()
Y_tambahan = tambahan[ASPEK].to_numpy()


def cv_aspek(sumber):
    prob = np.zeros(Y_dev.shape)
    for k, (latih_idx, uji_idx) in enumerate(lipatan_aspek.split(dev)):
        teks_latih = dev['teks_ringan'].iloc[latih_idx].tolist() + tambahan['teks_ringan'].tolist()
        Y_latih = np.concatenate([Y_dev[latih_idx], Y_tambahan]).astype(np.float32)
        model, tok = latih(sumber, teks_latih, Y_latih, multi_label=True, epoch=EPOCH_ASPEK,
                           bobot=bobot_aspek(Y_latih), seed=SEED + k)
        prob[uji_idx] = prediksi(model, tok, dev['teks_ringan'].iloc[uji_idx].tolist(), multi_label=True)
        del model
        bersihkan()
        print(f'  lipatan {k + 1}/5 selesai')
    return prob


prob_aspek = muat_atau_hitung(f'oof_aspek_{DASAR_TERPILIH}', lambda: cv_aspek(MODEL_DASAR[DASAR_TERPILIH]))
pred_aspek = (prob_aspek >= 0.5).astype(int)

p, r, f, _ = precision_recall_fscore_support(Y_dev, pred_aspek, average=None, zero_division=0)
bawah, atas = interval_bootstrap(Y_dev, pred_aspek, macro_f1_aspek)
print(f'{DASAR_TERPILIH} multi-label: macro-F1 {macro_f1_aspek(Y_dev, pred_aspek):.3f} (CI 95% {bawah:.2f}-{atas:.2f}), '
      f'micro-F1 {f1_score(Y_dev, pred_aspek, average="micro", zero_division=0):.3f}')

# Baseline dibaca dari 03_baseline.ipynb (dev, lipatan yang sama)
baseline_aspek = pd.read_parquet(FOLDER_PROCESSED / 'ringkasan_dev_baseline_aspek.parquet')
if (baseline_aspek['versi_label'] != VERSI_LABEL).any():
    print('PERINGATAN: ringkasan baseline dari versi label lain; jalankan ulang 03_baseline.ipynb lalu sel ini')
baseline_aspek = baseline_aspek.set_index('model')
display(pd.DataFrame({
    **{nama: baseline_aspek.loc[nama, [f'f1_{a}' for a in ASPEK]].to_numpy(dtype=float) for nama in baseline_aspek.index},
    'IndoBERT presisi': p, 'IndoBERT recall': r, 'IndoBERT F1': f,
    'jumlah_dev': Y_dev.sum(axis=0),
}, index=ASPEK).round(2))
print('Macro-F1 baseline (dev):',
      '; '.join(f"{n} {b['macro_f1']:.3f} ({b['ci_95']})" for n, b in baseline_aspek.iterrows()))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  lipatan 1/5 selesai


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  lipatan 2/5 selesai


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  lipatan 3/5 selesai


... (2 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  lipatan 5/5 selesai
oof_aspek_IndoBERT: selesai dalam 2.6 menit
IndoBERT multi-label: macro-F1 0.797 (CI 95% 0.77-0.82), micro-F1 0.811


,Kata kunci,TF-IDF + OvR LR (C=16),IndoBERT presisi,IndoBERT recall,IndoBERT F1,jumlah_dev
transaksi,0.66,0.78,0.75,0.90,0.82,203
keamanan,0.63,0.66,0.60,0.72,0.65,50
akun,0.80,0.81,0.81,0.85,0.83,137
error_performa,0.78,0.81,0.76,0.90,0.82,196
cs,0.74,0.79,0.69,0.83,0.75,52
biaya_promo,0.76,0.84,0.80,0.82,0.81,87
paylater,0.89,0.88,0.81,0.96,0.88,70
kemudahan,0.64,0.59,0.82,0.80,0.81,45


Macro-F1 baseline (dev): Kata kunci 0.737 (0.70-0.76); TF-IDF + OvR LR (C=16) 0.770 (0.74-0.80)


In [11]:
PATH_MODEL_ASPEK = FOLDER_MODEL / 'indobert_aspek'
NAMA_ASPEK = f'{DASAR_TERPILIH} multi-label'

if model_terbaru(PATH_MODEL_ASPEK) and sudah_diprediksi('aspek', NAMA_ASPEK):
    print('Model aspek final dan prediksi test sudah ada, dilewati')
else:
    mulai = time.time()
    Y_gold_train = gold_train[ASPEK].to_numpy().astype(np.float32)
    model, tok = latih(MODEL_DASAR[DASAR_TERPILIH], gold_train['teks_ringan'].tolist(), Y_gold_train,
                       multi_label=True, epoch=EPOCH_ASPEK, bobot=bobot_aspek(Y_gold_train))
    simpan_prediksi_test('aspek', NAMA_ASPEK, prediksi(model, tok, test['teks_ringan'].tolist(), multi_label=True),
                         KOLOM_PROB_ASPEK)
    simpan_model(model, tok, PATH_MODEL_ASPEK, {
        'nama': NAMA_ASPEK, 'dasar': MODEL_DASAR[DASAR_TERPILIH], 'aspek': ASPEK, 'ambang': 0.5,
        'jumlah_latih': len(gold_train), 'macro_f1_dev_5fold': round(macro_f1_aspek(Y_dev, pred_aspek), 3),
        'epoch': EPOCH_ASPEK, 'lr': LR, 'max_len': MAX_LEN,
    })
    print(f'Disimpan ke models/indobert_aspek/ ({(time.time() - mulai) / 60:.1f} menit)')
    del model
    bersihkan()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Disimpan ke models/indobert_aspek/ (0.7 menit)


# 6.13 Ringkasan sentimen (dev)

In [12]:
# Baris baseline dibaca dari 03_baseline.ipynb (dev yang sama, lipatan yang sama)
baseline_sentimen = pd.read_parquet(FOLDER_PROCESSED / 'ringkasan_dev_baseline_sentimen.parquet')
if (baseline_sentimen['versi_label'] != VERSI_LABEL).any():
    print('PERINGATAN: ringkasan baseline dari versi label lain; jalankan ulang 03_baseline.ipynb lalu sel ini')
kolom = ['model', 'macro_f1', 'ci_95', 'f1_neg', 'f1_net', 'f1_pos', 'catatan']
pd.concat([
    baseline_sentimen[kolom].assign(kelompok='baseline'),
    pd.DataFrame(hasil_sentimen)[kolom].assign(kelompok='IndoBERT'),
]).set_index(['kelompok', 'model'])


macro_f1      ci_95  f1_neg  f1_net  \
kelompok model                                                                  
baseline Rating bintang                      0.523  0.49-0.55   0.709   0.174   
         Panjang ulasan saja                 0.534  0.50-0.57   0.780   0.144   
         Lexicon InSet                       0.467  0.43-0.51   0.801   0.183   
         TF-IDF + LR, label rating (C=1)     0.599  0.57-0.63   0.798   0.179   
         TF-IDF + LR, label gold (C=1)       0.817  0.77-0.85   0.936   0.603   
IndoBERT IndoBERT, label gold                0.859  0.82-0.89   0.958   0.655   
         IndoBERTweet, label gold            0.831  0.79-0.87   0.951   0.612   
         IndoBERT, label rating              0.558  0.53-0.59   0.679   0.157   
         Dua tahap (rating -> gold)          0.868  0.83-0.90   0.958   0.678   

                                          f1_pos  \
kelompok model                                     
baseline Rating bintang                    0.685   
         Panjang ulasan saja               0.679   
         Lexicon InSet                     0.418   
         TF-IDF + LR, label rating (C=1)   0.820   
         TF-IDF + LR, label gold (C=1)     0.912   
IndoBERT IndoBERT, label gold              0.964   
         IndoBERTweet, label gold          0.930   
         IndoBERT, label rating            0.840   
         Dua tahap (rating -> gold)        0.969   

                                                                            catatan  
kelompok model                                                                       
baseline Rating bintang                   bukan model; label rating vs label manual  
         Panjang ulasan saja                              LR 1 fitur, 5-fold di dev  
         Lexicon InSet                                        ambang dipilih di dev  
         TF-IDF + LR, label rating (C=1)             397,434 ulasan berlabel rating  
         TF-IDF + LR, label gold (C=1)               5-fold di dev + sampel terarah  
IndoBERT IndoBERT, label gold                        5-fold di dev + sampel terarah  
         IndoBERTweet, label gold                    5-fold di dev + sampel terarah  
         IndoBERT, label rating                          20,000 ulasan per aplikasi  
         Dua tahap (rating -> gold)                  5-fold di dev + sampel terarah

## 6.14 Error analysis (dev)

In [14]:
pred_terbaik = label_dari_prob(prob_oof[TERBAIK])
salah = dev.assign(prediksi=pred_terbaik, yakin=prob_oof[TERBAIK].max(axis=1))
salah = salah[salah['sentimen'] != salah['prediksi']]
print(f'{TERBAIK}: {len(salah)} kesalahan dari {len(dev)} ulasan dev')
display(pd.crosstab(salah['sentimen'], salah['prediksi'], rownames=['gold'], colnames=['prediksi']))

for _, r in salah.sample(n=min(50, len(salah)), random_state=SEED).sort_values(['sentimen', 'prediksi']).iterrows():
    print(f"{r['id_label']} | rating {r['rating']} | gold {r['sentimen']} -> {r['prediksi']} ({r['yakin']:.2f}) | "
          f"{r['teks_ringan'][:200]}")

Dua tahap (rating -> gold): 50 kesalahan dari 855 ulasan dev


prediksi,neg,net,pos
gold,,,
neg,0,19,5
net,18,0,1
pos,7,0,0


G0756 | rating 5 | gold neg -> net (0.70) | tap saldo kok tidak muncul ya
G0571 | rating 3 | gold neg -> net (0.97) | harus pakai ktp
P115 | rating 5 | gold neg -> net (0.96) | cara tampilkan dana instant sama dana cicil bagaimana nih tolong di jelaskan
G0812 | rating 4 | gold neg -> net (0.87) | syaa sudah lama pakai dana tapi dana cicil nya belum ada
G0401 | rating 2 | gold neg -> net (0.72) | aplikasi dana saya kok tidak bisa di buka ya
G0874 | rating 4 | gold neg -> net (0.97) | pinjaman dana cicil gbisa2
G0427 | rating 2 | gold neg -> net (0.53) | kenapa tidak ada saldo ovo nya saya
P043 | rating 4 | gold neg -> net (0.88) | jangan terlalu berharap
P094 | rating 4 | gold neg -> net (0.99) | semoga akun ovo ku bisa untu transaksi jual beli pulsa dan melakukan transaksi transfer ke bank lain kalau sudah bisa saya kasih bintang 5 terima kasih
G0568 | rating 2 | gold neg -> net (0.50) | s s s s s s s s s
G0815 | rating 4 | gold neg -> net (0.90) | tolong tingkatkan keamana aplikasi da